In [1]:
import numpy as np
import pandas as pd
import pickle

In [2]:
data_path = '../data/data.pickle'
comp_path = '../data/composition_12_mol.pkl'

In [3]:
# base data file
with open(data_path,'rb') as f:
    df_full = pickle.load(f)
# composition file
with open(comp_path,'rb') as f:
    composition = pickle.load(f)

In [4]:
# df_full['GP_33_2017.mdb'].keys()

In [5]:
description = df_full['GP_33_2017.mdb']['TJ_PROPERTY']
# description = df_full['GP_33_2017.mdb']['TJ_GLASSINFO']

In [6]:
description

,GLASS_ID,PROP_ID,BRANCH_NO,PROP_UNIT_DESC_EN,PROP_FIG,PROP_CONDITION,IMAGE_NAME,PROP_UNIT_01,PROP_UNIT_02,PROP_UNIT_03,PROP_UNIT_04,PROP_FIG_SI
0,000001,1303,1,m2/s,9.800000e-14,309K,NaN,0,1,0,0,9.800000e-14
1,000001,1380,1,NaN,NaN,NaN,NaN,0,0,0,0,NaN
2,000002,1303,1,m2/s,1.700000e-13,337K,NaN,0,1,0,0,1.700000e-13
3,000002,1380,1,NaN,NaN,NaN,NaN,0,0,0,0,NaN
4,000003,1303,1,m2/s,1.600000e-13,329K,NaN,0,1,0,0,1.600000e-13
...,...,...,...,...,...,...,...,...,...,...,...,...
1323250,363839,2240,1,NaN,3.500000e-01,"350nm, 1mmt",NaN,1,1,1,1,3.500000e-01
1323251,363839,2251,1,NaN,2.000000e-01,"500nm, 1mmt",NaN,1,1,1,1,2.000000e-01
1323252,363839,3041,1,S/cm,2.950000e+02,NaN,NaN,1,0,1,1,2.950000e+04
1323253,363839,3045,1,eV,8.190000e-01,NaN,NaN,1,0,0,1,7.862400e+04


In [7]:
propty = {
    'Density:': 510,
    'YoungsModulus:': 540,
    'ShearModulus:': 50,
    'VickersHardness:': 180,
    'PoissonRatio:': 60,
    'FractureToughness:': 160,
    'CrystallizationTemp:': 1014,
    'ElectricConduct:': 3012,
    'DielectricConst:': 3174,
    'GlassTransitionTg:': 1140,
    'TSofP:': 1116,
    'TAnnealingP:': 1119,
    'TStrainP:': 1123,
    'ExpansionCoeff:': 1020,
    'LiquidusTemperature:': 1011,
    'SofP:': 1118,
    'ThermalShockResistance:': 1390,
    'TensileStrength:': 120,
    'CompressiveStrength:': 140,
    'FlexuralStrength:': 102,
    'BulkModulus:': 70,
    'RefractiveIndex:': 2010,
    'AbbeValue:': 2051,
    'ThermalConductivity:': 1633,
    'DiffusionCoeff:': 1305,
    'ActivationEnergy:': 1306,
    'TransmittanceUV:': 2200,
    'TransmittanceIR:': 2209,
    'TransmittanceVisible:': 2210,
    'TransmittanceSolar:': 2212,
    'DCBreakdownVoltage:': 3188,
    'SpecificHeat:': 1603
}


In [8]:
mask = None
print(description.keys())
for value in propty.values():
    prop_id = f"{value:04d}"  # format the number to be 4 digits long with leading zeros
    if mask is None:
        mask = (description['PROP_ID'] == prop_id)
    else:
        mask |= (description['PROP_ID'] == prop_id)

# Now 'mask' contains the combined condition for all values in the dictionary.
# print(mask)

Index(['GLASS_ID', 'PROP_ID', 'BRANCH_NO', 'PROP_UNIT_DESC_EN', 'PROP_FIG',
       'PROP_CONDITION', 'IMAGE_NAME', 'PROP_UNIT_01', 'PROP_UNIT_02',
       'PROP_UNIT_03', 'PROP_UNIT_04', 'PROP_FIG_SI'],
      dtype='object')


In [9]:
# pid = 510
# 510 density, 1140 Tg, 2010 ri, 2051 abbe
# mask = description['PROP_ID'] == str(pid).zfill(4)
#mask = (description['PROP_ID'] == "0510") | (description['PROP_ID'] == "1140") | (description['PROP_ID'] == "2010") | (description['PROP_ID'] == "2051")
ppty = description[mask] # [['GLASS_ID','PROP_FIG_SI']].dropna()
composition['GLASS_ID'] = composition.index.astype(int)
ppty['GLASS_ID'] = ppty['GLASS_ID'].astype(int)
comp_density_df = pd.merge(composition, ppty, on='GLASS_ID', how='inner')

/home/scai/phd/aiz217586/.conda/envs/discomat/lib/python3.7/site-packages/ipykernel_launcher.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  import sys


In [10]:
comp_pii_df = pd.read_csv('../data/subset_interglad.csv')
comp_pii_df = comp_pii_df.dropna(subset=['GLASS_ID'])
comp_pii_df['GLASS_ID'] = comp_pii_df['GLASS_ID'].astype(int)

/home/scai/phd/aiz217586/.conda/envs/discomat/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3258: DtypeWarning: Columns (539) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [11]:
comp_density_pii_df = pd.merge(comp_density_df, comp_pii_df, on='GLASS_ID', how='inner')

In [12]:
comp_density_pii_df.to_csv('../data/combined_property_interglad.csv', index=False)